# Full Analysis with ML Features

This notebook processes all 17 basins from `data/processed/*.tif` and generates a complete ML-ready dataset.

## Features Computed

1. **CouplingAnalyzer metrics**: `touching`, `contact_px`, `size1_px`, `size2_px`
2. **LengthwiseAsymmetryAnalyzer metrics**: `L_1`, `L_2`, `delta_L`
3. **GeometricFeaturesAnalyzer metrics**: `orientation_diff_deg`, `headhead_dist_m`, `headhead_dist_norm`, `apex_angle_deg`, `strahler_order_diff`

## Outputs

- Per-basin intermediate CSVs: `data/results/{basin}/full_features.csv`
- Combined master dataset: `data/results/master_dataset_v2.csv`

## Reference

> Goren, L. and Shelef, E.: Channel concavity controls planform complexity of branching drainage networks, Earth Surf. Dynam., 12, 1347-1369, https://doi.org/10.5194/esurf-12-1347-2024, 2024.

---
## 1. Setup & Configuration

In [19]:
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import topotoolbox as tt3
from tqdm import tqdm

from channel_heads import (
    # Core analysis
    CouplingAnalyzer,
    LengthwiseAsymmetryAnalyzer,
    GeometricFeaturesAnalyzer,
    first_meet_pairs_for_outlet,
    outlet_node_ids_from_streampoi,
    # Dataset generation
    generate_labeled_dataset,
    filter_hard_negatives,
    # Configuration
    get_basin_config,
)
from channel_heads.config import RESULTS_DIR, PROCESSED_DIR
from channel_heads.basin_config import LOCAL_TO_PAPER_BASIN

warnings.filterwarnings('ignore', category=RuntimeWarning)

print(f"Processed DEMs directory: {PROCESSED_DIR}")
print(f"Results directory: {RESULTS_DIR}")

Processed DEMs directory: /Users/guypi/Projects/channel-heads/data/cropped_DEMs
Results directory: /Users/guypi/Projects/channel-heads/data/results


In [20]:
# === CONFIGURATION ===

# Stream network parameters
STREAM_THRESHOLD = 300  # Minimum drainage area (pixels) for stream initiation
CONNECTIVITY = 8        # Basin contact detection connectivity (4 or 8)

# Negative sampling parameters
NEGATIVE_RATIO = 3.0    # Target negative:positive ratio (3:1)
RANDOM_SEED = 42        # For reproducibility

# Hard negative filtering thresholds (from filter_hard_negatives)
HARD_NEG_MAX_L_RATIO = 3.0     # Max L_sum ratio vs median positive
HARD_NEG_MAX_DIST_RATIO = 5.0  # Max distance ratio vs median positive

# Output paths
MASTER_OUTPUT_PATH = RESULTS_DIR / "master_dataset_v2.csv"

print("Configuration:")
print(f"  Stream threshold: {STREAM_THRESHOLD} pixels")
print(f"  Connectivity: {CONNECTIVITY}")
print(f"  Target neg:pos ratio: {NEGATIVE_RATIO}:1")
print(f"  Output: {MASTER_OUTPUT_PATH}")

Configuration:
  Stream threshold: 300 pixels
  Connectivity: 8
  Target neg:pos ratio: 3.0:1
  Output: /Users/guypi/Projects/channel-heads/data/results/master_dataset_v2.csv


In [21]:
# Discover all DEMs in data/processed/ and map to basin configs
# Basin name mapping: DEM filename -> config name
DEM_TO_BASIN = {
    "CalnAlpine_strm_crop.tif": "calnalpine",
    "Daqing_strm_crop.tif": "daqing",
    "Finisterre_strm_crop.tif": "finisterre",
    "Humboldt_strm_crop.tif": "humboldt",
    "Inyo_strm_crop.tif": "inyo",
    "Kammanasie_strm_crop.tif": "kammanasie",
    "Luliang_strm_crop.tif": "luliang",
    "Panamint_strm_crop.tif": "panamint",
    "Sakhalin_strm_crop.tif": "sakhalin",
    "SierraMadre_strm_crop.tif": "sierramadre",
    "SierraNevadaSpain_strm_crop.tif": "sierranevadaspain",
    "SierradelValleFertil_strm_crop.tif": "vallefertil",
    "Taiwan_strm_crop.tif": "taiwan",
    "Toano_strm_crop.tif": "toano",
    "Troodos_strm_crop.tif": "troodos",
    "Tsugaru_strm_crop.tif": "tsugaru",
    "Yoro_strm_crop.tif": "yoro",
}

# Build list of basins to process
BASINS = []
missing_dems = []
missing_configs = []

for dem_file, basin_name in sorted(DEM_TO_BASIN.items()):
    dem_path = PROCESSED_DIR / dem_file
    
    if not dem_path.exists():
        missing_dems.append(dem_file)
        continue
    
    try:
        config = get_basin_config(basin_name)
        BASINS.append({
            "name": basin_name,
            "dem_path": dem_path,
            "dem_file": dem_file,
            "z_th": config["z_th"],
            "lat": config["lat"],
            "full_name": config["full_name"],
        })
    except KeyError as e:
        missing_configs.append((basin_name, str(e)))

print(f"Found {len(BASINS)} basins to process:")
print()
for b in BASINS:
    print(f"  {b['name']:20s} z_th={b['z_th']:5d}m  lat={b['lat']:7.2f}  {b['full_name']}")

if missing_dems:
    print(f"\nMissing DEMs: {missing_dems}")
if missing_configs:
    print(f"\nMissing configs: {missing_configs}")

Found 17 basins to process:

  calnalpine           z_th= 1700m  lat=  39.69  Clan Alpine Mountains, Nevada
  daqing               z_th= 1200m  lat=  40.71  Daquing Shan, China
  finisterre           z_th=  400m  lat=  -5.95  Finisterre Range, Papua New Guinea
  humboldt             z_th= 1450m  lat=  40.52  Humboldt Range, Nevada
  inyo                 z_th= 1200m  lat=  36.71  Inyo Mountains, California
  kammanasie           z_th=  630m  lat= -33.62  Kammanassie Mountains, South Africa
  luliang              z_th= 1100m  lat=  39.27  Lüliang Mountains, China
  panamint             z_th=  800m  lat=  36.17  Panamint Range, California
  sakhalin             z_th=   60m  lat=  47.07  Sakhalin Mountains, Russia
  sierramadre          z_th=  380m  lat=  17.52  Sierra Madre del Sur, Mexico
  sierranevadaspain    z_th= 1200m  lat=  37.05  Sierra Nevada, Spain
  vallefertil          z_th= 1050m  lat= -30.44  Sierra del Valle Fértil, Argentina
  taiwan               z_th=   80m  lat=  23.47 

---
## 2. Processing Functions

In [22]:
def process_outlet(
    outlet_id: int,
    s,
    coupling_an: CouplingAnalyzer,
    asym_an: LengthwiseAsymmetryAnalyzer,
    geom_an: GeometricFeaturesAnalyzer,
) -> pd.DataFrame | None:
    """
    Process a single outlet: find pairs and compute all features.
    
    Returns DataFrame with merged features, or None if no valid pairs.
    """
    # Find channel head pairs that first meet at confluences
    pairs_at_confluence, basin_heads = first_meet_pairs_for_outlet(s, outlet_id)
    
    if not pairs_at_confluence:
        return None
    
    total_pairs = sum(len(p) for p in pairs_at_confluence.values())
    if total_pairs == 0:
        return None
    
    # 1. Coupling analysis: touching detection
    coupling_df = coupling_an.evaluate_pairs_for_outlet(outlet_id, pairs_at_confluence)
    if coupling_df.empty:
        return None
    
    # 2. Lengthwise asymmetry: L_1, L_2, delta_L
    asym_df = asym_an.evaluate_pairs_for_outlet(outlet_id, pairs_at_confluence)
    
    # 3. Geometric features: confluence angle, orientation, distance, tortuosity
    geom_df = geom_an.evaluate_pairs_for_outlet(
        outlet_id, pairs_at_confluence, asymmetry_df=asym_df
    )
    
    # 4. Generate labeled dataset (merges all + adds y column)
    labeled_df = generate_labeled_dataset(coupling_df, asym_df, geom_df)
    
    return labeled_df

In [23]:
def process_basin(basin_config: dict, pbar_position: int = 1) -> tuple[pd.DataFrame | None, dict]:
    """
    Process a complete basin: load DEM, create network, analyze all outlets.
    
    Returns:
        - DataFrame with all results, or None if failed
        - Stats dictionary
    """
    basin_name = basin_config["name"]
    dem_path = basin_config["dem_path"]
    z_th = basin_config["z_th"]
    lat = basin_config["lat"]
    
    stats = {
        "basin": basin_name,
        "n_outlets": 0,
        "n_outlets_with_pairs": 0,
        "n_pairs": 0,
        "n_touching": 0,
        "n_not_touching": 0,
        "time_s": 0.0,
        "error": None,
    }
    
    start_time = time.time()
    
    try:
        # Load DEM
        dem = tt3.read_tif(str(dem_path))
        
        # Apply elevation threshold mask
        dem.z[dem.z < z_th] = np.nan
        
        # Create flow direction and stream network
        fd = tt3.FlowObject(dem)
        s = tt3.StreamObject(fd, threshold=STREAM_THRESHOLD)
        
        # Get all outlets
        outlets = outlet_node_ids_from_streampoi(s)
        stats["n_outlets"] = len(outlets)
        
        if len(outlets) == 0:
            stats["error"] = "No outlets found"
            stats["time_s"] = time.time() - start_time
            return None, stats
        
        # Initialize analyzers
        coupling_an = CouplingAnalyzer(
            fd, s, dem,
            connectivity=CONNECTIVITY,
            threshold=STREAM_THRESHOLD,
        )
        asym_an = LengthwiseAsymmetryAnalyzer(s, dem, lat=lat)
        node_orders = s.streamorder(method="strahler")
        geom_an = GeometricFeaturesAnalyzer(s, dem, lat=lat, node_orders=node_orders)
        
        # Process each outlet with progress bar
        all_results = []
        
        outlet_pbar = tqdm(
            outlets,
            desc=f"    {basin_name} outlets",
            leave=False,
            position=pbar_position,
        )
        
        for outlet_id in outlet_pbar:
            try:
                outlet_df = process_outlet(
                    outlet_id, s, coupling_an, asym_an, geom_an
                )
                
                if outlet_df is not None and not outlet_df.empty:
                    all_results.append(outlet_df)
                    stats["n_outlets_with_pairs"] += 1
                    
            except Exception as e:
                # Skip failed outlets, continue with others
                continue
            finally:
                # Clear mask cache between outlets to manage memory
                coupling_an.clear_cache()
        
        outlet_pbar.close()
        
        # Combine all outlet results
        if not all_results:
            stats["error"] = "No pairs found in any outlet"
            stats["time_s"] = time.time() - start_time
            return None, stats
        
        df_basin = pd.concat(all_results, ignore_index=True)
        df_basin["basin"] = basin_name
        
        # Compute statistics
        stats["n_pairs"] = len(df_basin)
        stats["n_touching"] = int(df_basin["y"].sum())
        stats["n_not_touching"] = len(df_basin) - stats["n_touching"]
        stats["time_s"] = time.time() - start_time
        
        return df_basin, stats
        
    except Exception as e:
        stats["error"] = str(e)
        stats["time_s"] = time.time() - start_time
        return None, stats

In [24]:
def stratified_subsample_negatives(
    df: pd.DataFrame,
    target_ratio: float = 3.0,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Subsample negatives using stratified sampling per basin.
    
    - Keeps ALL positive samples (touching=True, y=1)
    - Subsamples negatives to achieve target negative:positive ratio
    - Uses stratified sampling to maintain basin distribution
    
    Parameters
    ----------
    df : pd.DataFrame
        Labeled dataset with 'y' and 'basin' columns.
    target_ratio : float
        Desired negative:positive ratio.
    random_state : int
        Random seed.
        
    Returns
    -------
    pd.DataFrame
        Subsampled dataset.
    """
    if df.empty:
        return df
    
    rng = np.random.default_rng(random_state)
    
    positives = df[df["y"] == 1].copy()
    negatives = df[df["y"] == 0].copy()
    
    n_pos = len(positives)
    n_neg_current = len(negatives)
    n_neg_target = int(n_pos * target_ratio)
    
    print(f"    Positives: {n_pos:,}")
    print(f"    Negatives (current): {n_neg_current:,}")
    print(f"    Negatives (target): {n_neg_target:,}")
    
    if n_neg_current <= n_neg_target:
        print(f"    -> Keeping all negatives (already below target)")
        return df
    
    # Stratified sampling by basin
    basin_neg_counts = negatives.groupby("basin").size()
    basin_proportions = basin_neg_counts / basin_neg_counts.sum()
    basin_targets = (basin_proportions * n_neg_target).round().astype(int)
    
    # Adjust to exactly match target
    diff = n_neg_target - basin_targets.sum()
    if diff != 0:
        # Add/remove from basins with largest counts
        adjustment_order = basin_neg_counts.sort_values(ascending=False).index
        for i, basin in enumerate(adjustment_order):
            if diff == 0:
                break
            if diff > 0:
                basin_targets[basin] += 1
                diff -= 1
            elif diff < 0 and basin_targets[basin] > 0:
                basin_targets[basin] -= 1
                diff += 1
    
    # Sample from each basin
    sampled_negatives = []
    for basin, n_target in basin_targets.items():
        basin_negs = negatives[negatives["basin"] == basin]
        if len(basin_negs) <= n_target:
            sampled_negatives.append(basin_negs)
        else:
            indices = rng.choice(len(basin_negs), size=n_target, replace=False)
            sampled_negatives.append(basin_negs.iloc[indices])
    
    df_sampled_neg = pd.concat(sampled_negatives, ignore_index=True)
    
    # Combine positives + sampled negatives
    result = pd.concat([positives, df_sampled_neg], ignore_index=True)
    result = result.sort_values(["basin", "outlet", "confluence"], ignore_index=True)
    
    actual_ratio = len(df_sampled_neg) / n_pos if n_pos > 0 else 0
    print(f"    -> Final: {n_pos:,} pos + {len(df_sampled_neg):,} neg (ratio={actual_ratio:.2f}:1)")
    
    return result

---
## 3. Process All Basins

Process each basin, compute all features, save intermediate results.

In [25]:
all_basin_results = []
all_stats = []

total_start = time.time()

print("=" * 80)
print(f"PROCESSING {len(BASINS)} BASINS")
print("=" * 80)

basin_pbar = tqdm(BASINS, desc="Basins", position=0)

for basin_config in basin_pbar:
    basin_name = basin_config["name"]
    basin_pbar.set_description(f"Basin: {basin_name}")
    
    print(f"\n{'─'*80}")
    print(f"Processing: {basin_name.upper()}")
    print(f"  DEM: {basin_config['dem_file']}")
    print(f"  z_th: {basin_config['z_th']}m, lat: {basin_config['lat']:.2f}°")
    
    # Process basin
    df_basin, stats = process_basin(basin_config, pbar_position=1)
    all_stats.append(stats)
    
    if df_basin is not None and not df_basin.empty:
        # Save intermediate result
        basin_output_dir = RESULTS_DIR / basin_name
        basin_output_dir.mkdir(parents=True, exist_ok=True)
        intermediate_path = basin_output_dir / "full_features.csv"
        df_basin.to_csv(intermediate_path, index=False)
        
        touch_pct = stats['n_touching'] / stats['n_pairs'] * 100
        print(f"  ✓ {stats['n_pairs']:,} pairs ({stats['n_outlets_with_pairs']}/{stats['n_outlets']} outlets)")
        print(f"    Touching: {stats['n_touching']:,} ({touch_pct:.1f}%)")
        print(f"    Not touching: {stats['n_not_touching']:,} ({100-touch_pct:.1f}%)")
        print(f"    Time: {stats['time_s']:.1f}s")
        print(f"    Saved: {intermediate_path}")
        
        all_basin_results.append(df_basin)
    else:
        print(f"  ✗ FAILED: {stats.get('error', 'Unknown error')}")
        print(f"    Time: {stats['time_s']:.1f}s")
    
    # Memory cleanup between basins
    gc.collect()

basin_pbar.close()

total_time = time.time() - total_start
print(f"\n{'='*80}")
print(f"PROCESSING COMPLETE - Total time: {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"{'='*80}")

PROCESSING 17 BASINS


Basin: calnalpine:   0%|          | 0/17 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────────────────────────
Processing: CALNALPINE
  DEM: CalnAlpine_strm_crop.tif
  z_th: 1700m, lat: 39.69°


  ✓ 7 pairs (4/28 outlets)
    Touching: 4 (57.1%)
    Not touching: 3 (42.9%)
    Time: 0.1s
    Saved: /Users/guypi/Projects/channel-heads/data/results/calnalpine/full_features.csv


Basin: daqing:   6%|▌         | 1/17 [00:00<00:03,  5.15it/s]    

...

  ✓ 171 pairs (19/41 outlets)
    Touching: 60 (35.1%)
    Not touching: 111 (64.9%)
    Time: 0.7s
    Saved: /Users/guypi/Projects/channel-heads/data/results/troodos/full_features.csv

────────────────────────────────────────────────────────────────────────────────
Processing: TSUGARU
  DEM: Tsugaru_strm_crop.tif
  z_th: 30m, lat: 40.97°


Basin: yoro:  88%|████████▊ | 15/17 [03:59<00:54, 27.15s/it]   

  ✓ 12 pairs (3/21 outlets)
    Touching: 6 (50.0%)
    Not touching: 6 (50.0%)
    Time: 0.0s
    Saved: /Users/guypi/Projects/channel-heads/data/results/tsugaru/full_features.csv

────────────────────────────────────────────────────────────────────────────────
Processing: YORO
  DEM: Yoro_strm_crop.tif
  z_th: 130m, lat: 35.28°


Basin: yoro: 100%|██████████| 17/17 [03:59<00:00, 14.11s/it]

  ✓ 5 pairs (3/12 outlets)
    Touching: 5 (100.0%)
    Not touching: 0 (0.0%)
    Time: 0.0s
    Saved: /Users/guypi/Projects/channel-heads/data/results/yoro/full_features.csv

PROCESSING COMPLETE - Total time: 239.8s (4.0 min)


In [26]:
# Display processing statistics table
df_stats = pd.DataFrame(all_stats)

print("\nPROCESSING STATISTICS")
print("-" * 80)

successful = df_stats[df_stats["error"].isna()]
failed = df_stats[df_stats["error"].notna()]

print(f"Successful: {len(successful)}/{len(df_stats)} basins")
if len(failed) > 0:
    print(f"Failed: {len(failed)} basins")
    for _, row in failed.iterrows():
        print(f"  - {row['basin']}: {row['error']}")

print(f"\nTotals:")
print(f"  Outlets processed: {df_stats['n_outlets'].sum():,}")
print(f"  Outlets with pairs: {df_stats['n_outlets_with_pairs'].sum():,}")
print(f"  Total pairs: {df_stats['n_pairs'].sum():,}")
print(f"  Total touching: {df_stats['n_touching'].sum():,}")
print(f"  Total not touching: {df_stats['n_not_touching'].sum():,}")
print(f"  Total time: {df_stats['time_s'].sum():.1f}s")

# Show table
print("\n" + "-" * 80)
display_cols = ["basin", "n_outlets", "n_outlets_with_pairs", "n_pairs", "n_touching", "n_not_touching", "time_s"]
print(df_stats[display_cols].to_string(index=False))


PROCESSING STATISTICS
--------------------------------------------------------------------------------
Successful: 17/17 basins

Totals:
  Outlets processed: 1,258
  Outlets with pairs: 396
  Total pairs: 10,496
  Total touching: 2,315
  Total not touching: 8,181
  Total time: 238.8s

--------------------------------------------------------------------------------
            basin  n_outlets  n_outlets_with_pairs  n_pairs  n_touching  n_not_touching     time_s
       calnalpine         28                     4        7           4               3   0.112269
           daqing         28                     7       13          10               3   0.068946
       finisterre        192                    65     1821         415            1406  22.925486
         humboldt         42                     7       13           8               5   0.082418
             inyo         28                    10       15           7               8   0.073144
       kammanasie         38          

---
## 4. Combine Basins & Apply Negative Sampling

In [27]:
if not all_basin_results:
    raise RuntimeError("No basin results to combine!")

# Combine all basins
df_combined = pd.concat(all_basin_results, ignore_index=True)

print(f"Combined dataset: {df_combined.shape[0]:,} rows x {df_combined.shape[1]} columns")
print(f"\nColumns: {list(df_combined.columns)}")
print(f"\nClass balance (before filtering):")
print(f"  y=1 (touching): {(df_combined['y']==1).sum():,}")
print(f"  y=0 (not touching): {(df_combined['y']==0).sum():,}")

Combined dataset: 10,496 rows x 23 columns

Columns: ['outlet', 'confluence', 'head_1', 'head_2', 'touching', 'contact_px', 'size1_px', 'size2_px', 'skipped_prefilter', 'y', 'L_1', 'L_2', 'delta_L', 'orientation_diff_deg', 'headhead_dist_m', 'headhead_dist_norm', 'apex_angle_deg', 'strahler_order_diff', 'proximity_mean_m', 'proximity_max_m', 'proximity_profile_norm', 'qc_flags', 'basin']

Class balance (before filtering):
  y=1 (touching): 2,315
  y=0 (not touching): 8,181


In [28]:
# Step 1: Apply hard negative filtering
print("Step 1: Applying hard negative filtering...")
print(f"  max_L_ratio: {HARD_NEG_MAX_L_RATIO}")
print(f"  max_dist_ratio: {HARD_NEG_MAX_DIST_RATIO}")

df_filtered = filter_hard_negatives(
    df_combined,
    max_L_ratio=HARD_NEG_MAX_L_RATIO,
    max_dist_ratio=HARD_NEG_MAX_DIST_RATIO,
)

n_removed = len(df_combined) - len(df_filtered)
print(f"\n  Before: {len(df_combined):,} rows")
print(f"  After:  {len(df_filtered):,} rows")
print(f"  Removed: {n_removed:,} trivial negatives ({n_removed/len(df_combined)*100:.1f}%)")
print(f"\n  Class balance after filtering:")
print(f"    y=1: {(df_filtered['y']==1).sum():,}")
print(f"    y=0: {(df_filtered['y']==0).sum():,}")

Step 1: Applying hard negative filtering...
  max_L_ratio: 3.0
  max_dist_ratio: 5.0

  Before: 10,496 rows
  After:  5,922 rows
  Removed: 4,574 trivial negatives (43.6%)

  Class balance after filtering:
    y=1: 2,315
    y=0: 3,607


In [29]:
# Step 2: Stratified subsampling to target ratio
print(f"\nStep 2: Stratified subsampling negatives (target ratio: {NEGATIVE_RATIO}:1)...")

df_master = stratified_subsample_negatives(
    df_filtered,
    target_ratio=NEGATIVE_RATIO,
    random_state=RANDOM_SEED,
)

print(f"\nFinal class balance:")
n_pos = (df_master['y'] == 1).sum()
n_neg = (df_master['y'] == 0).sum()
print(f"  y=1 (positive/touching): {n_pos:,}")
print(f"  y=0 (negative/not touching): {n_neg:,}")
print(f"  Actual ratio: {n_neg/n_pos:.2f}:1")


Step 2: Stratified subsampling negatives (target ratio: 3.0:1)...
    Positives: 2,315
    Negatives (current): 3,607
    Negatives (target): 6,945
    -> Keeping all negatives (already below target)

Final class balance:
  y=1 (positive/touching): 2,315
  y=0 (negative/not touching): 3,607
  Actual ratio: 1.56:1


---
## 5. Save Master Dataset

In [30]:
# Ensure results directory exists
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save master dataset
df_master.to_csv(MASTER_OUTPUT_PATH, index=False)

file_size_mb = MASTER_OUTPUT_PATH.stat().st_size / 1e6

print(f"Master dataset saved!")
print(f"  Path: {MASTER_OUTPUT_PATH}")
print(f"  Shape: {df_master.shape}")
print(f"  Size: {file_size_mb:.2f} MB")

Master dataset saved!
  Path: /Users/guypi/Projects/channel-heads/data/results/master_dataset_v2.csv
  Shape: (5922, 23)
  Size: 1.45 MB


---
## 6. Dataset Summary & Quality Report

In [31]:
print("=" * 80)
print("FINAL DATASET SUMMARY")
print("=" * 80)

print(f"\nTotal rows: {len(df_master):,}")
print(f"Total columns: {len(df_master.columns)}")
print(f"Basins: {df_master['basin'].nunique()}")

print("\n" + "-" * 40)
print("CLASS BALANCE")
print("-" * 40)
y_counts = df_master["y"].value_counts().sort_index()
for label, count in y_counts.items():
    pct = count / len(df_master) * 100
    label_str = "touching" if label == 1 else "not touching"
    print(f"  y={label} ({label_str}): {count:,} ({pct:.1f}%)")
print(f"  Ratio (neg:pos): {y_counts.get(0, 0) / max(y_counts.get(1, 1), 1):.2f}:1")

FINAL DATASET SUMMARY

Total rows: 5,922
Total columns: 23
Basins: 17

----------------------------------------
CLASS BALANCE
----------------------------------------
  y=0 (not touching): 3,607 (60.9%)
  y=1 (touching): 2,315 (39.1%)
  Ratio (neg:pos): 1.56:1


In [32]:
print("\n" + "-" * 40)
print("PER-BASIN STATISTICS")
print("-" * 40)

basin_summary = df_master.groupby("basin").agg(
    total=("y", "count"),
    touching=("y", "sum"),
    touch_ratio=("y", "mean"),
).round(3)
basin_summary["not_touching"] = basin_summary["total"] - basin_summary["touching"]
basin_summary["touching"] = basin_summary["touching"].astype(int)
basin_summary["not_touching"] = basin_summary["not_touching"].astype(int)
basin_summary = basin_summary[["total", "touching", "not_touching", "touch_ratio"]]
basin_summary = basin_summary.sort_values("total", ascending=False)

print(basin_summary.to_string())
print(f"\nTotal: {basin_summary['total'].sum():,}")


----------------------------------------
PER-BASIN STATISTICS
----------------------------------------
                   total  touching  not_touching  touch_ratio
basin                                                        
taiwan              2729      1054          1675        0.386
sierramadre         1517       576           941        0.380
finisterre          1090       415           675        0.381
troodos              154        60            94        0.390
vallefertil           80        46            34        0.575
luliang               73        33            40        0.452
sakhalin              68        28            40        0.412
panamint              52        22            30        0.423
sierranevadaspain     48        23            25        0.479
toano                 27        12            15        0.444
kammanasie            19         6            13        0.316
inyo                  15         7             8        0.467
humboldt              13    

In [33]:
print("\n" + "-" * 40)
print("MISSING VALUES")
print("-" * 40)

# Feature columns (ML features)
feature_cols = [
    # Coupling
    "contact_px", "size1_px", "size2_px",
    # Asymmetry
    "L_1", "L_2", "delta_L",
    # Geometric
    "orientation_diff_deg",
    "headhead_dist_m", "headhead_dist_norm",
    "apex_angle_deg", "strahler_order_diff",
    "proximity_mean_m", "proximity_max_m", "proximity_profile_norm",
]

missing_data = []
for col in feature_cols:
    if col in df_master.columns:
        n_missing = df_master[col].isna().sum()
        pct = n_missing / len(df_master) * 100
        missing_data.append({"feature": col, "missing": n_missing, "pct": pct})
    else:
        missing_data.append({"feature": col, "missing": "COLUMN NOT FOUND", "pct": 100.0})

df_missing = pd.DataFrame(missing_data)
print(df_missing.to_string(index=False))

# Summary
with_missing = df_missing[(df_missing["pct"] > 0) & (df_missing["missing"] != "COLUMN NOT FOUND")]
if len(with_missing) > 0:
    print(f"\n{len(with_missing)} features have missing values")
else:
    print("\nNo missing values in ML features!")


----------------------------------------
MISSING VALUES
----------------------------------------
               feature  missing       pct
            contact_px        0  0.000000
              size1_px     2857 48.243837
              size2_px     2857 48.243837
                   L_1        0  0.000000
                   L_2        0  0.000000
               delta_L        0  0.000000
  orientation_diff_deg        0  0.000000
       headhead_dist_m        0  0.000000
    headhead_dist_norm        0  0.000000
        apex_angle_deg        0  0.000000
   strahler_order_diff        0  0.000000
      proximity_mean_m        0  0.000000
       proximity_max_m        0  0.000000
proximity_profile_norm        0  0.000000

2 features have missing values


In [34]:
print("\n" + "-" * 40)
print("FEATURE STATISTICS")
print("-" * 40)

existing_features = [c for c in feature_cols if c in df_master.columns]
stats_df = df_master[existing_features].describe().T
stats_df = stats_df[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]].round(3)

print(stats_df.to_string())


----------------------------------------
FEATURE STATISTICS
----------------------------------------


                         count      mean       std      min       25%       50%       75%        max
contact_px              5922.0    16.531    25.041    0.000     0.000     0.000    33.000    146.000
size1_px                3065.0   338.752    51.402  300.000   305.000   317.000   351.000    725.000
size2_px                3065.0   338.532    50.722  300.000   305.000   317.000   352.000    685.000
L_1                     5922.0  5395.617  5381.213   88.535  1975.535  3945.210  7387.849  68263.497
L_2                     5922.0  5761.610  5692.430   76.291  2073.597  4338.483  7837.703  69622.900
delta_L                 5922.0     0.834     0.530    0.000     0.378     0.789     1.250      1.978
orientation_diff_deg    5922.0    46.427    37.197    0.000    17.857    38.320    66.177    180.000
headhead_dist_m         5922.0  3474.528  1889.709   88.535  2128.581  3015.382  4445.341  10182.912
headhead_dist_norm      5922.0     0.384     0.166    0.017     0.262     0.378     0.499  

In [35]:
if "qc_flags" in df_master.columns:
    print("\n" + "-" * 40)
    print("QC FLAGS SUMMARY")
    print("-" * 40)
    
    has_flags = df_master["qc_flags"].notna() & (df_master["qc_flags"] != "")
    n_with_flags = has_flags.sum()
    pct_with_flags = n_with_flags / len(df_master) * 100
    
    print(f"Rows with QC flags: {n_with_flags:,} ({pct_with_flags:.1f}%)")
    
    if n_with_flags > 0:
        all_flags = df_master.loc[has_flags, "qc_flags"].str.split(",").explode()
        flag_counts = all_flags.value_counts().head(10)
        print("\nTop 10 flags:")
        for flag, count in flag_counts.items():
            print(f"  {flag}: {count:,}")


----------------------------------------
QC FLAGS SUMMARY
----------------------------------------
Rows with QC flags: 155 (2.6%)

Top 10 flags:
  orient1:short_path: 81
  orient2:short_path: 79


In [36]:
print("\n" + "-" * 40)
print("DATASET PREVIEW")
print("-" * 40)

# Show first few rows
df_master.head(10)


----------------------------------------
DATASET PREVIEW
----------------------------------------


,outlet,confluence,head_1,head_2,touching,contact_px,size1_px,size2_px,skipped_prefilter,y,...,orientation_diff_deg,headhead_dist_m,headhead_dist_norm,apex_angle_deg,strahler_order_diff,proximity_mean_m,proximity_max_m,proximity_profile_norm,qc_flags,basin
0,0,13,34,167,True,11,323.0,300.0,False,1,...,19.281625,2776.396966,0.588806,35.450866,0.0,1626.764918,2776.396966,0.585927,,yoro
1,0,53,34,120,False,0,382.0,411.0,False,0,...,46.634634,2261.379689,0.712135,58.371332,1.0,1289.459627,2261.379689,0.570209,,luliang
2,0,112,120,156,True,35,411.0,318.0,False,1,...,65.132571,586.523493,0.506304,64.592282,0.0,285.063966,586.523493,0.486023,,luliang
3,0,400,1502,1798,True,19,311.0,344.0,False,1,...,30.663975,2540.608097,0.185809,14.804226,1.0,2079.893303,2746.937888,0.757168,,sakhalin
4,1,8,61,134,False,0,306.0,301.0,False,0,...,152.857181,3092.205526,0.661579,99.462322,3.0,1758.057833,3092.205526,0.568545,,finisterre
5,1,8,134,390,True,29,301.0,301.0,False,1,...,22.500000,2024.023776,0.361275,52.206057,3.0,1763.501354,2552.243240,0.690961,,finisterre
6,1,8,134,963,False,0,NaN,NaN,True,0,...,55.066286,3652.934768,0.423789,51.508956,3.0,2987.552977,4304.140342,0.694111,,finisterre
7,1,8,134,1718,False,0,NaN,NaN,True,0,...,9.735610,5444.779904,0.518143,44.215175,3.0,3587.304176,5444.779904,0.658852,,finisterre
8,1,24,111,118,True,24,300.0,315.0,False,1,...,22.890987,2094.081385,0.320015,30.411081,1.0,1310.356654,2094.081385,0.625743,,troodos
9,1,24,111,223,False,0,300.0,449.0,False,0,...,25.437887,2555.137912,0.339809,44.510304,1.0,1416.190321,2555.137912,0.554252,,troodos


---
## Notes

### Feature Descriptions

**Target:**
- `y`: Binary label (1=touching/coupled, 0=not touching)

**Identifiers:**
- `basin`: Basin name for train/test splitting
- `outlet`, `confluence`, `head_1`, `head_2`: Node IDs

**Coupling features:**
- `touching`: Boolean (same as y)
- `contact_px`: Contact boundary length (pixels)
- `size1_px`, `size2_px`: Basin sizes (pixels)

**Lengthwise asymmetry (Goren & Shelef 2024):**
- `L_1`, `L_2`: Flow path lengths to confluence (meters)
- `delta_L`: Normalized asymmetry = 2|L_1 - L_2| / (L_1 + L_2), range [0, 2]

**Geometric features:**
- `orientation_diff_deg`: Difference in initial flow directions [0, 180]
- `headhead_dist_m`: Euclidean head-to-head distance (meters)
- `headhead_dist_norm`: headhead_dist_m / (L_1 + L_2)
- `apex_angle_deg`: Angle at confluence formed by straight lines to heads
- `strahler_order_diff`: Strahler stream order difference between branches

### Next Steps

1. Load `data/results/master_dataset_v2.csv`
2. Use `basin` column for leave-one-basin-out cross-validation
3. Handle missing values (imputation or feature selection)
4. See `notebooks/02_xgboost_classification.ipynb` for model training